<a href="https://colab.research.google.com/github/manika-lamba/SP26-LIS4_5693/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Topic Modeling

In this lab assignment, we will learn to perform topic modeling using `gensim` library.

*Note: Before starting this lab assignment, please complete the Introduction to Topic Modeling notebook*

## Learning Objectives

In this exercise, you will:

- Load and process the assigned data file
- Run LDA algorithm
- Visualize topic results

This exercise builds directly on concepts from discussed in the precursor notebook on "Introduction to Topic Modeling".

## Install and Import Packages

We will be using the Gensim library to create our topic model and the PyLDAVis library to visualize it. We will also need to install NLTK, or the Natural Language Toolkit, in order to get a list of stop words.



In [2]:
!pip install gensim
!pip install pyldavis
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.7 MB/s eta 0:00:00


Once you have installed NLTK, you will need to download the list of English stop words. You can do so with the following command:

In [4]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Now that we have our libraries installed correctly, we can import everything.

In [17]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
import pandas as pd
from nltk.corpus import stopwords
import string
import gensim.corpora as corpora
from gensim.models import LdaModel
import pyLDAvis.gensim_models
pyLDAvis.enable_notebook()

This will import the requisite model from Gensim. For this notebook, we will be using the LdaModel class. This class allows us to create an LDA model. Before we can populate our model, however, we must first load and clean our data.

In [18]:
import requests
import io

url = "https://raw.githubusercontent.com/manika-lamba/SP26-LIS4_5693/refs/heads/main/lab-assignments/lab-5/trc.csv"
response = requests.get(url)
response.raise_for_status()
text = response.text

Let's save it as a DataFrame using df variable that holds pandas DataFrame and see how it looks!

In [19]:
import pandas as pd

df = pd.read_csv(io.StringIO(text))
print(df.head())

   ObjectId      Last                  First  \
0         1     AARON            Thabo Simon   
1         2    ABBOTT              Montaigne   
2         3   ABRAHAM  Nzaliseko Christopher   
3         4  ABRAHAMS         Achmat Fardiel   
4         5  ABRAHAMS       Annalene Mildred   

                                         Description      Place      Yr  \
0  An ANCYL member who was shot and severely inju...   Bethulie  1991.0   
1  A member of the SADF who was severely injured ...    Messina  1987.0   
2  A COSAS supporter who was kicked and beaten wi...  Mdantsane  1985.0   
3  Was shot and blinded in one eye by members of ...    Athlone  1985.0   
4  Was shot and injured by members of the SAP in ...  Robertson  1990.0   

  Homeland           Province        Long        Lat           HRV  \
0      NaN  Orange Free State    25.97552 -30.503290  shoot|injure   
1      NaN          Transvaal   30.039597 -22.351308        injure   
2   Ciskei  Cape of Good Hope  27.6708791 -32.9586

In [20]:
df = df[["Last", "First", "Description"]]
df

,Last,First,Description
0,AARON,Thabo Simon,An ANCYL member who was shot and severely inju...
1,ABBOTT,Montaigne,A member of the SADF who was severely injured ...
2,ABRAHAM,Nzaliseko Christopher,A COSAS supporter who was kicked and beaten wi...
3,ABRAHAMS,Achmat Fardiel,Was shot and blinded in one eye by members of ...
4,ABRAHAMS,Annalene Mildred,Was shot and injured by members of the SAP in ...
...,...,...,...
20829,XUZA,Mandla,Was severely injured when he was stoned by a f...
20830,YAKA,Mbangomuni,An IFP supporter and acting induna who was sho...
20831,YALI,Khayalethu,"Was shot by members of the SAP in Lingelihle, ..."
20832,YALO,Bikiwe,An IFP supporter whose house and possessions w...


Our goal in this section will be to model all the descriptions of violence in the TRC Volume 7 Final Report. We will, therefore, grab all documents and place them into a list.

In [21]:
docs = df.Description.tolist()
print(docs[0])

An ANCYL member who was shot and severely injured by SAP members at Lephoi, Bethulie, Orange Free State (OFS) on 17 April 1991. Police opened fire on a gathering at an ANC supporter's house following a dispute between two neighbours, one of whom was linked to the ANC and the other to the SAP and a councillor.


## Cleaning Documents

Now that we have our documents, let’s go ahead and load up our stop words. These will be the words that we remove from our documents.

In [23]:
stop_words = stopwords.words('english')

Next, we need a function to clean our documents. The purpose of the function below is to take a single document as an input and return a cleaned sequence of words with no punctuation or stop words.

Adding custom stopwords

In [48]:
custom_stopwords = ['mr.', '16', 'june', '30', '7']
stop_words.extend(custom_stopwords)
print(f"New stopwords added: {custom_stopwords}")

New stopwords added: ['mr.', '16', 'june', '30', '7']


In [49]:
def clean_doc(doc):
    no_punct = ''
    for c in doc:
        if c not in string.punctuation:
            no_punct = no_punct+c
    # with list comprehension
    # no_punct = ''.join([c for c in doc if c not in string.punctuation])

    words = no_punct.lower().split()

    final_words = []
    for word in words:
        if word not in stop_words:
            final_words.append(word)

    # with list comprehension
    # final_words = [word for word in words if word not in stop_words]

    return final_words


In [50]:
cleaned_docs = [clean_doc(doc) for doc in docs]
id2word = corpora.Dictionary(cleaned_docs)
corpus = [id2word.doc2bow(cleaned_doc) for cleaned_doc in cleaned_docs]

print("Cleaning process, ID-Word Index, and Corpus have been regenerated with new stopwords.")

cleaned_with_custom_stopwords = clean_doc(docs[0])
print("\nOriginal document (first one):")
print(docs[0])
print("\nCleaned document with custom stopwords:")
print(cleaned_with_custom_stopwords)

Cleaning process, ID-Word Index, and Corpus have been regenerated with new stopwords.

Original document (first one):
An ANCYL member who was shot and severely injured by SAP members at Lephoi, Bethulie, Orange Free State (OFS) on 17 April 1991. Police opened fire on a gathering at an ANC supporter's house following a dispute between two neighbours, one of whom was linked to the ANC and the other to the SAP and a councillor.

Cleaned document with custom stopwords:
['ancyl', 'member', 'shot', 'severely', 'injured', 'sap', 'members', 'lephoi', 'bethulie', 'orange', 'free', 'state', 'ofs', '17', 'april', '1991', 'police', 'opened', 'fire', 'gathering', 'anc', 'supporters', 'house', 'following', 'dispute', 'two', 'neighbours', 'one', 'linked', 'anc', 'sap', 'councillor']


Let’s take a look and see what this looks like now when we run it over our first document.



With our function created, we can now process all our documents. In the line be below, we will convert all documents into a list called cleaned_docs. This will be our documents that are now represented as a sequence of words.

In [26]:
cleaned_docs = [clean_doc(doc) for doc in docs]

## Create ID-Word Index

Remember, an LDA topic model cannot look at words, rather it must look at numbers in order for the algorithm to work. This means that we need to convert all words into numbers. We can do this with a bag-of-words approach. In this approach, we create an ID-Word Index. This is essentially a dictionary where each unique word has a unique number. The dictionary is sorted alphabetically.

In [27]:
id2word = corpora.Dictionary(cleaned_docs)

With the ID-Word Index created, let’s query it and see what word maps to index number `250`.

In [28]:
id2word[250]

'bmw'

As we can see, it is `BMW`, a political organization in South Africa. Now that we have our dictionary, we can convert all our documents into a sequence of numbers, rather than words. We can do this via the `doc2bow()` method.

In [29]:
corpus = [id2word.doc2bow(cleaned_doc) for cleaned_doc in cleaned_docs]

The `corpus` object now contains all the documents but represented as a bag-of-words index, rather than as a sequence of words. This is the precise data that our LDA model will expect.

In [30]:
print(corpus[0])

[(0, 1), (1, 1), (2, 2), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 2), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1)]


In order to see what these numbers correspond to, let’s take a look at our first document and map each number to the ID-Word Index.

In [31]:
for num in corpus[0]:
    num = num[0]
    print(f"{num}\t{id2word[num]}")

0	17
1	1991
2	anc
3	ancyl
4	april
5	bethulie
6	councillor
7	dispute
8	fire
9	following
10	free
11	gathering
12	house
13	injured
14	lephoi
15	linked
16	member
17	members
18	neighbours
19	ofs
20	one
21	opened
22	orange
23	police
24	sap
25	severely
26	shot
27	state
28	supporters
29	two


## Creating LDA Topic Model

With our `corpus` now prepared,, we can pass it to our LDA Model. We need have a lot of parameters that we can use here, but three things are mandatory:

- `corpus` which will be our corpus object

- `id2word` which will be our ID-Word Index

- `num_topics` which will be the number of topics we want the model to find.

Now, we can train our topic model.

In [32]:
lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=100)

Training a model may take several seconds to minutes. The more documents you have, the longer it will take. Once the model is trained, we can begin to analyze the model.

## Analyze a Document

If we want to get the topics for each document in our corpus, we can do so via `get_document_topics()`. This will take a single argument, the `corpus` object. It will return a list of topics and a score for each topic.

In [33]:
topics = lda_model.get_document_topics(corpus)
len(topics)

20834

We can analyze the topics for our first document.

In [40]:
for topic_id, score in topics[0]:
    print(f"({topic_id}, {float(score)})")

(0, 0.042218055576086044)
(8, 0.0745592936873436)
(18, 0.06086849048733711)
(39, 0.21957650780677795)
(48, 0.28521353006362915)
(56, 0.03858255594968796)
(58, 0.08404681086540222)
(59, 0.02601402811706066)
(77, 0.0440683476626873)
(82, 0.09757835417985916)


These topics were assigned to the following document:

In [38]:
print(docs[0])

An ANCYL member who was shot and severely injured by SAP members at Lephoi, Bethulie, Orange Free State (OFS) on 17 April 1991. Police opened fire on a gathering at an ANC supporter's house following a dispute between two neighbours, one of whom was linked to the ANC and the other to the SAP and a councillor.


Let’s now print off some of the key words for each of the top-3 topics. For each topic, we will print off the top-10 words with the `get_topic_terms()` method which will take two arguments: the topic number and the amount of words to return.

In [45]:
for topic in topics[0][:3]:
    terms = lda_model.get_topic_terms(topic[0], 10)
    print(f"({topic[0]}, {float(topic[1])})")
    for num in terms:
        num = num[0]
        print(num, id2word[num])
    print()

(0, 0.042237065732479095)
736 1984
196 boycott
584 rent
6 councillor
1135 councillors
1068 returned
17 members
24 sap
26 shot
2006 vaal

(8, 0.07459807395935059)
558 set
545 alight
408 accused
115 burnt
2550 party
7 dispute
96 death
12 house
2 anc
28 supporters

(18, 0.0608186274766922)
26 shot
37 transvaal
17 members
821 gang
239 local
85 dead
16 member
1389 sebokeng
1081 19
3 ancyl



## Analyze the Topic Model

It is often difficult to gauge the quality of a topic model by simply looking at raw output. For these reasons, it is useful to be able to study a model in its entirety. We can do this with PyLDAVis, a library that was designed to display in 2-dimensional space the topics of a topic model and the key words associated with each topic.

In [46]:
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word, mds="mmds", R=30)

In [47]:
vis

PreparedData(topic_coordinates=              x         y  topics  cluster      Freq
topic                                               
31    -0.083599 -0.389300       1        1  4.374016
92    -0.137870 -0.361365       2        1  3.556112
57    -0.156726 -0.443091       3        1  3.467502
94    -0.105853 -0.280206       4        1  2.918172
72     0.348096  0.243074       5        1  2.847979
...         ...       ...     ...      ...       ...
86     0.313422  0.085641      96        1  0.287221
58     0.132040 -0.168910      97        1  0.283022
50    -0.277546  0.275073      98        1  0.224889
99    -0.281949  0.418949      99        1  0.183989
70    -0.208621  0.250047     100        1  0.179910

[100 rows x 5 columns], topic_info=           Term          Freq         Total  Category  logprob  loglift
347         ifp   8034.000000   8034.000000   Default  30.0000  30.0000
349       natal   4375.000000   4375.000000   Default  29.0000  29.0000
28   supporters  10746.000000  10746.000000   Default  28.0000  28.0000
348     kwazulu   4463.000000   4463.000000   Default  27.0000  27.0000
358      durban   4233.000000   4233.000000   Default  26.0000  26.0000
..          ...           ...           ...       ...      ...      ...
2           anc     11.266889  11208.178186  Topic100  -4.2877  -0.5821
26         shot      9.721072   8729.005864  Topic100  -4.4353  -0.4796
4         april      7.695725   2395.702318  Topic100  -4.6689   0.5797
17      members      8.785883   8048.637431  Topic100  -4.5364  -0.4996
29          two      6.098289   2366.402940  Topic100  -4.9016   0.3594

[4663 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
697       4  0.076619             1
697       6  0.026050             1
697       8  0.029115             1
697       9  0.067425             1
697      12  0.139447             1
...     ...       ...           ...
1577     35  0.984979         ‘red’
2855     85  0.781630  ‘sharpeville
3006     52  0.403621   ‘terrorist’
518      63  0.670349      ‘whites’
520      99  0.965123            ’s

[10955 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[32, 93, 58, 95, 73, 44, 35, 46, 31, 23, 80, 11, 42, 69, 90, 68, 60, 56, 19, 52, 10, 47, 6, 63, 54, 40, 49, 37, 81, 53, 22, 14, 39, 64, 76, 61, 30, 12, 41, 24, 18, 75, 4, 62, 88, 72, 28, 3, 66, 89, 9, 99, 5, 38, 50, 78, 96, 84, 98, 20, 74, 97, 83, 1, 48, 79, 86, 77, 45, 91, 43, 27, 55, 36, 25, 65, 8, 13, 16, 29, 33, 70, 92, 57, 15, 82, 67, 94, 34, 2, 26, 7, 21, 17, 85, 87, 59, 51, 100, 71])

Now, we can see all 100 topics plotted in two-dimensional space and we can study the degree to which the words in each topic are unique to that topic when compared with the corpus as as whole.

While useful as a first step to exploring data, we have more powerful ways of clustering documents today. As we will see other libraries and approaches that leverage state of the art language models are perhaps better suited to finding and extracting topics from your corpus in Week-12.